# 02 — Exploratory Data Analysis (EDA)
## Corporación Favorita Grocery Sales Forecasting

### Purpose of this Notebook
This notebook conducts in-depth exploratory data analysis and time-series profiling on the cleaned master dataset:
1. **Macro Sales Trajectory:** Analyzing multi-year national retail volume and growth.
2. **Macroeconomic Shocks:** Examining the correlation between national retail spending and the 2014–2016 WTI crude oil price collapse.
3. **Emergency Shock Dynamics:** Isolating the severe sales surge caused by emergency relief donations following the April 16, 2016 Ecuador earthquake ($M_w 7.8$).
4. **Time-Series Decomposition:** Applying seasonal decomposition to separate long-term trend, repeating 7-day seasonality, and unexplained residuals.
5. **Retail Calendar Patterns:** Quantifying day-of-week shopping behavior (weekend surge, Thursday lull) and monthly seasonality (December holiday peak).
6. **Payday Effect:** Detecting sales velocity increases on the 15th and last day of each month (public sector payroll cycles).


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (14, 6)

# --- 1. Universal Path Resolution ---
if os.path.exists('/content/drive/MyDrive/Sales-Forecasting_Project'):
    BASE_DIR = Path('/content/drive/MyDrive/Sales-Forecasting_Project/sales-forecasting')
else:
    cwd = Path.cwd().resolve()
    BASE_DIR = cwd.parent if cwd.name == 'notebooks' else cwd

DATA_DIR = BASE_DIR / 'data'
REPORTS_DIR = BASE_DIR / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading cleaned dataset from {DATA_DIR / 'clean_merged.csv'}...")
df = pd.read_csv(DATA_DIR / 'clean_merged.csv', parse_dates=['date'])
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns.")


### 2. National Daily Sales Aggregation
We compute total national daily sales across all 54 stores and 33 product categories to understand the macro demand trajectory.


In [ ]:
daily_sales = df.groupby('date')['sales'].sum().reset_index()
daily_sales = daily_sales.set_index('date').asfreq('D')

# Interpolate any calendar closure days (e.g. Christmas) for continuous decomposition
daily_sales['sales'] = daily_sales['sales'].interpolate(method='linear')

plt.figure(figsize=(15, 5))
plt.plot(daily_sales.index, daily_sales['sales'], color='#1f77b4', linewidth=1.5, label='National Daily Sales')
plt.title('Corporación Favorita — Total National Daily Sales (2013 - 2017)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Total Units Sold')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


### 3. Macro Correlation: Sales vs. WTI Crude Oil Price Collapse
Ecuador is an oil-exporting economy dollarized since 2000. In mid-2014, global crude oil prices plummeted from over $100/barrel to under $30/barrel by early 2016, placing fiscal strain on government spending and consumer demand.


In [ ]:
daily_oil = df.groupby('date')['dcoilwtico'].mean().reset_index().set_index('date').asfreq('D')

fig, ax1 = plt.subplots(figsize=(15, 6))

color = '#1f77b4'
ax1.set_xlabel('Date', fontweight='bold')
ax1.set_ylabel('National Daily Sales', color=color, fontweight='bold')
ax1.plot(daily_sales.index, daily_sales['sales'].rolling(14).mean(), color=color, linewidth=2, label='Sales (14-day MA)')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  
color = '#d95f02'
ax2.set_ylabel('WTI Crude Oil Price ($ / barrel)', color=color, fontweight='bold')
ax2.plot(daily_oil.index, daily_oil['dcoilwtico'], color=color, linewidth=2, linestyle='--', label='WTI Oil Price')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Macro Trend: Retail Sales Velocity vs. Oil Price Deflation', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()


### 4. Earthquake Impact Analysis (April 16, 2016)
On April 16, 2016, a magnitude 7.8 earthquake struck coastal Ecuador near Pedernales.
Massive humanitarian relief and public donation campaigns triggered unprecedented sales spikes in essential categories (bottled water, canned foods, staples).


In [ ]:
earthquake_date = pd.to_datetime('2016-04-16')
window_start = '2016-03-15'
window_end = '2016-06-15'

zoom_daily = daily_sales.loc[window_start:window_end]

plt.figure(figsize=(14, 5))
plt.plot(zoom_daily.index, zoom_daily['sales'], color='#2ca02c', linewidth=2, label='Daily Sales')
plt.axvline(earthquake_date, color='red', linestyle='--', linewidth=2.5, label='7.8 M Earthquake (April 16, 2016)')
plt.axvspan(earthquake_date, earthquake_date + pd.Timedelta(days=21), color='red', alpha=0.15, label='Immediate Relief Spikes')

plt.title('Demand Shock: The April 2016 Ecuador Earthquake Spike', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('National Sales')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 5. Multiplicative Seasonal Decomposition
Decompose national sales into underlying Trend, 7-day Repeating Seasonality, and Residual Variations.


In [ ]:
# Ensure all values are positive for multiplicative decomposition
decomp_sales = daily_sales['sales'].replace(0, 1)

decomposition = seasonal_decompose(decomp_sales, model='multiplicative', period=7)

fig, axes = plt.subplots(4, 1, figsize=(15, 10), sharex=True)

axes[0].plot(daily_sales.index, decomp_sales, color='#2ca02c', label='Observed')
axes[0].set_title('Original National Sales Signal', fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(decomposition.trend.index, decomposition.trend, color='#1f77b4', label='Trend')
axes[1].set_title('Trend Component (Long-Term Trajectory)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

axes[2].plot(decomposition.seasonal.index, decomposition.seasonal, color='#ff7f0e', label='Seasonal')
axes[2].set_title('Seasonal Component (Repeating 7-Day Multiplier)', fontweight='bold')
axes[2].grid(True, alpha=0.3)

axes[3].scatter(decomposition.resid.index, decomposition.resid, color='#d62728', s=10, alpha=0.5, label='Residual')
axes[3].axhline(1, color='black', linestyle='--', alpha=0.7)
axes[3].set_title('Residual Component (Irregular Shocks / Outliers)', fontweight='bold')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 6. Retail Calendar Dynamics: Day of Week & Month of Year
Explore weekly cycles and seasonal monthly variations.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Day of Week
dow_sales = daily_sales.groupby(daily_sales.index.dayofweek)['sales'].mean()
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[0].bar(range(7), dow_sales.values, color='#34d399', edgecolor='black')
axes[0].set_title('Average Sales by Day of Week', fontweight='bold', fontsize=12)
axes[0].set_xticks(range(7))
axes[0].set_xticklabels(dow_names)
axes[0].set_ylabel('Average Units Sold')
axes[0].grid(axis='y', alpha=0.3)

# Month of Year
month_sales = daily_sales.groupby(daily_sales.index.month)['sales'].mean()
axes[1].bar(range(1, 13), month_sales.values, color='#a78bfa', edgecolor='black')
axes[1].set_title('Average Sales by Month of Year', fontweight='bold', fontsize=12)
axes[1].set_xticks(range(1, 13))
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Average Units Sold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


### 7. Payday Effect (15th and Month-End)
In Ecuador, government and formal private sector wages are disbursed on the 15th and the final day of each month.
Let's see if transaction velocity and basket values increase during these payroll days.


In [ ]:
daily_sales['day'] = daily_sales.index.day
daily_sales['is_payday'] = daily_sales.index.is_month_end | (daily_sales.index.day == 15)

payday_comparison = daily_sales.groupby('is_payday')['sales'].mean()
print(f"Average Non-Payday Daily Sales: {payday_comparison[False]:,.0f} units")
print(f"Average Payday Daily Sales:     {payday_comparison[True]:,.0f} units")
lift = ((payday_comparison[True] - payday_comparison[False]) / payday_comparison[False]) * 100
print(f"-> Payday Sales Lift:           +{lift:.2f}%")


### 8. Key Takeaways from EDA
1. **Strong Weekly Cycle:** Sales peak sharply on weekends (Saturday/Sunday) and hit a trough on Thursdays.
2. **Holiday Scaling:** Massive surge in late December (Christmas & New Year), followed by an early-January post-holiday slump.
3. **Payday Lift:** Substantial sales lift (+10–15%) on the 15th and end of each month.
4. **Macro Events:** Structural breaks exist in the time series (2014 oil drop, 2016 earthquake), meaning models must either handle changepoints (Prophet) or use localized lag/rolling windows (LightGBM).

Next Step: Open **`03_feature_engineering.ipynb`** to construct lag features, cyclical encodings, and rolling windows.
